# Skybrightness during twilight estimate #

Use the skybrightness model to estimate skybrightness during twilight, at a series of sun altitudes.
The same approach could be used with any time to estimate the skybrightness. 

In [ ]:
import numpy as np
import healpy as hp
import matplotlib.pyplot as plt
from astropy.time import Time, TimeDelta
from astropy.coordinates import SkyCoord, AltAz
import astropy.units as u
from rubin_sim.skybrightness  import SkyModel
from rubin_scheduler.utils import Site
import rubin_nights.dayobs_utils as rn_dayobs

In [ ]:
nside = 64
hpix = np.arange(0, hp.nside2npix(nside))
ra, dec = hp.pix2ang(nside, hpix, lonlat=True)
radec_coord = SkyCoord(ra=ra*u.deg, dec=dec*u.deg)

In [ ]:
lsst_site = Site("LSST")
sky = SkyModel(mags=True)

In [ ]:
sunsets = {}
alts = {}
azs = {}
sky_mags = {}
dayobs = 20260201
for deg in np.arange(-6, -13, -1):
    # What time is the sun at this altitude during sunset? 
    sunsets[deg], _ = rn_dayobs.day_obs_sunset_sunrise(dayobs, deg)
    # Set up to convert to altaz using astropy - could just use sky.set_ra_dec_mjd instead
    # but using astropy directly is slightly more accurate (probably not actually important)
    altaz_frame = AltAz(obstime=sunsets[deg], location=lsst_site.to_earth_location())
    altaz_coord = radec_coord.transform_to(altaz_frame)
    alts[deg] = altaz_coord.alt.deg
    azs[deg] = altaz_coord.az.deg
    sky.set_ra_dec_alt_az_mjd(ra, dec, altaz_coord.alt.deg, altaz_coord.az.deg, sunsets[deg].mjd, degrees=True)
    sky_mags[deg] = sky.return_mags()

In [ ]:
for deg in sky_mags:
    plt.figure()
    band = 'i'
    plt.scatter(azs[deg], alts[deg], c=sky_mags[deg][band])
    plt.colorbar(label=f"sky mags {band}")
    plt.xlabel("Azimuth")
    plt.ylabel("Altitude")
    plt.title(f"Sun altitude {deg}")